# 4) Answer Generation with Verified Context Only
## Corrective RAG System — Step 4 of 4

This notebook combines all the previous steps into one complete pipeline: retrieve → grade → correct
→ generate an answer **only** from the chunks that passed the grading step (verified context), citing
sources, and honestly handling questions with no reliable answer.

This notebook covers:
- Generate answers using verified context only
- Show answers with sources
- Handle unanswered questions
- Reduce hallucination

> This notebook is self-contained, so we re-define the retrieval and correction layer explained in
> detail in `03_Retrieval_Grading_Correction`.


In [1]:
import os
import urllib.request
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma

load_dotenv()

PROJECT_ROOT = Path.cwd()
VECTORSTORE_DIR = PROJECT_ROOT / "vectorstore" / "chroma_db"
COLLECTION_NAME = "crag_course_docs"
EMBEDDING_MODEL = "nomic-embed-text"
LLM_MODEL = "llama3.2:3b"
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

try:
    urllib.request.urlopen(OLLAMA_BASE_URL, timeout=3)
except Exception as exc:
    raise RuntimeError(
        f"Could not reach Ollama at {OLLAMA_BASE_URL}. Make sure the Ollama app is running "
        "(it starts automatically after installation, or run 'ollama serve' manually)."
    ) from exc

if not VECTORSTORE_DIR.exists():
    raise FileNotFoundError("No vector store found - run notebook 02_Embeddings_VectorStore first.")

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=OLLAMA_BASE_URL)
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(VECTORSTORE_DIR),
)
llm = ChatOllama(model=LLM_MODEL, temperature=0, base_url=OLLAMA_BASE_URL)

print(f"Vector store loaded with {vectorstore._collection.count()} vectors")


Vector store loaded with 11 vectors


In [2]:
def retrieve(query: str, k: int = 4):
    return vectorstore.similarity_search(query, k=k)


class GradeDocument(BaseModel):
    """Binary relevance grade for a retrieved document."""

    binary_score: Literal["yes", "no"] = Field(
        description="'yes' if the document is relevant to the question, otherwise 'no'"
    )


GRADER_SYSTEM_PROMPT = (
    "You are a grader assessing the relevance of a retrieved document to a user question.\n"
    "If the document contains information that helps answer the question, grade it as relevant.\n"
    "This does not need to be a strict, exact match — the goal is to filter out clearly irrelevant "
    "or off-topic retrievals, not to be overly strict."
)

grader_llm = llm.with_structured_output(GradeDocument)


def grade_document(question: str, document_text: str) -> str:
    prompt = f"Retrieved document:\n\n{document_text}\n\nUser question: {question}"
    result = grader_llm.invoke(
        [
            {"role": "system", "content": GRADER_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ]
    )
    return result.binary_score


REWRITER_SYSTEM_PROMPT = (
    "You are a query re-writer that converts an input question into a better version optimized for "
    "vector store retrieval. Look at the input and try to reason about the underlying semantic intent, "
    "and make the question more specific and information-dense. Return only the rewritten question."
)


def rewrite_query(question: str) -> str:
    response = llm.invoke(
        [
            {"role": "system", "content": REWRITER_SYSTEM_PROMPT},
            {"role": "user", "content": f"Original question: {question}\n\nRewritten question:"},
        ]
    )
    return response.content.strip()


def corrective_retrieve(
    question: str,
    k: int = 4,
    relevance_threshold: float = 0.5,
    max_rewrites: int = 2,
):
    """Retrieve -> grade -> (rewrite -> retrieve again) until enough relevant context is found."""
    trace = []
    current_query = question
    best_relevant_docs: list = []

    for attempt in range(max_rewrites + 1):
        docs = retrieve(current_query, k=k)
        grades = [grade_document(question, doc.page_content) for doc in docs]
        relevant_docs = [doc for doc, grade in zip(docs, grades) if grade == "yes"]
        ratio = len(relevant_docs) / len(docs) if docs else 0.0

        trace.append(
            {
                "attempt": attempt,
                "query_used": current_query,
                "retrieved": len(docs),
                "relevant": len(relevant_docs),
                "relevance_ratio": round(ratio, 2),
            }
        )

        if len(relevant_docs) > len(best_relevant_docs):
            best_relevant_docs = relevant_docs

        if ratio >= relevance_threshold:
            return relevant_docs, trace

        if attempt < max_rewrites:
            current_query = rewrite_query(current_query)

    return best_relevant_docs, trace


## Step 1 — Strict, hallucination-resistant answer prompt

In [3]:
ANSWER_SYSTEM_PROMPT = (
    "You are a helpful assistant answering questions using ONLY the provided context.\n"
    "Rules:\n"
    "1. Use only information present in the context below - never rely on outside knowledge.\n"
    "2. If the context does not contain enough information to answer, say clearly that you don't "
    "have enough verified information to answer, instead of guessing.\n"
    "3. When you do answer, cite the source file(s) you used in square brackets, using the exact file "
    "name shown after 'Source:' above (for example, if you used a chunk whose source line says "
    "'Source: 01_rag_basics.txt', cite it as [01_rag_basics.txt]). Never write the literal placeholder "
    "text 'source.txt'.\n"
    "4. Be concise and directly answer the question."
)


def build_context(docs) -> str:
    blocks = [f"Source: {doc.metadata['source']}\n{doc.page_content}" for doc in docs]
    return "\n\n---\n\n".join(blocks)


def generate_answer(question: str, verified_docs) -> str:
    context = build_context(verified_docs)
    user_prompt = f"Context:\n\n{context}\n\nQuestion: {question}"
    response = llm.invoke(
        [
            {"role": "system", "content": ANSWER_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]
    )
    return response.content.strip()


## Step 2 — Full pipeline: retrieve → grade → correct → generate (or honestly refuse)

In [4]:
NO_ANSWER_MESSAGE = (
    "I don't have enough verified information in the available documents to confidently answer this "
    "question. Try rephrasing it, or add more documents to data/raw."
)


def answer_question(question: str, k: int = 4, relevance_threshold: float = 0.5, max_rewrites: int = 2):
    verified_docs, trace = corrective_retrieve(
        question, k=k, relevance_threshold=relevance_threshold, max_rewrites=max_rewrites
    )

    if not verified_docs:
        return {
            "question": question,
            "answer": NO_ANSWER_MESSAGE,
            "sources": [],
            "trace": trace,
        }

    answer = generate_answer(question, verified_docs)
    sources = sorted({doc.metadata["source"] for doc in verified_docs})

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "trace": trace,
    }


## Step 3 — Demo run

In [5]:
sample_questions = [
    "How does Corrective RAG reduce hallucination compared to standard RAG?",
    "What is the difference between Chroma and FAISS?",
    "What is the capital of France?",  # out-of-scope question - should be refused honestly
]

for q in sample_questions:
    result = answer_question(q)
    print("=" * 90)
    print(f"Q: {result['question']}")
    print(f"\nA: {result['answer']}")
    print(f"\nSources: {result['sources']}")
    print(f"Correction attempts: {len(result['trace'])}")
    print()


Q: How does Corrective RAG reduce hallucination compared to standard RAG?

A: [03_corrective_rag.txt] According to the text, Corrective RAG reduces hallucination by requiring a system evaluation step after retrieval. If too few chunks are relevant, the system rewrites the query and retrieves again, repeating this correction loop until sufficient context is found. This approach ensures that only verified, relevant context is used for answer generation, preventing silent hallucinations in standard RAG.

Sources: ['01_rag_basics.txt', '03_corrective_rag.txt']
Correction attempts: 1



Q: What is the difference between Chroma and FAISS?

A: [02_vector_databases.txt]

Chroma and FAISS differ in their nature. Chroma is an open-source, embedded vector database that can be run locally from a Python process, whereas FAISS is a library rather than a full database.

Sources: ['02_vector_databases.txt']
Correction attempts: 1



Q: What is the capital of France?

A: I don't have enough verified information to answer this question.

Sources: ['01_rag_basics.txt', '03_corrective_rag.txt']
Correction attempts: 1

